In [ ]:
# AI Assisted Code
# Genereated by Claude
# Adapted by Authors

import os

HEIGHT = 100

GWA_BASE_URL = "https://globalwindatlas.info/api/gis/country"

GWA_FILE = "gwa_25_countries_100m.csv"
MERGED_FILE = "gwa_plus_generation_capacity.csv"
FINAL_FILE = "wind.csv"

# temporary folder for downloads
DOWNLOAD_FOLDER = "gwa_downloads"
os.makedirs(DOWNLOAD_FOLDER, exist_ok=True)

COUNTRIES = {
    "CHN": "China",
    "USA": "United States",
    "DEU": "Germany",
    "IND": "India",
    "BRA": "Brazil",
    "GBR": "United Kingdom",
    "ESP": "Spain",
    "FRA": "France",
    "CAN": "Canada",
    "SWE": "Sweden",
    "TUR": "Türkiye",
    "AUS": "Australia",
    "MEX": "Mexico",
    "NLD": "Netherlands",
    "DNK": "Denmark",
    "POL": "Poland",
    "PRT": "Portugal",
    "GRC": "Greece",
    "IRL": "Ireland",
    "NOR": "Norway",
    "FIN": "Finland",
    "CHL": "Chile",
    "ZAF": "South Africa",
    "ITA": "Italy",
    "ARG": "Argentina",
}

print(f"{len(COUNTRIES)} Länder konfiguriert.")


In [ ]:
import time
import requests
import rasterio
import numpy as np
import pandas as pd

# function to download a file from a URL and save it to a specified path
def download_file(url, output_path):
    if os.path.exists(output_path) and os.path.getsize(output_path) > 1000:
        print(f"  Bereits heruntergeladen: {output_path}")
        return

    print(f"  Lade herunter:\n  {url}")
    response = requests.get(url, stream=True, timeout=(30, 300), allow_redirects=True)
    response.raise_for_status()

    total_size = int(response.headers.get("content-length", 0))
    downloaded = 0
    with open(output_path, "wb") as f:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                f.write(chunk)
                downloaded += len(chunk)
                if total_size > 0:
                    percent = downloaded / total_size * 100
                    print(f"\r  Fortschritt: {percent:.1f}%", end="")
    print()
    print(f"  Gespeichert: {output_path}")


def calculate_statistics(file_path, prefix):
    """ reads a raster in blocks, filters invalid/negative values, 
    and computes exact statistics from ALL valid cells (no sampling)."""
    print(f"  Verarbeite {file_path}")

    chunks = []
    with rasterio.open(file_path) as src:
        nodata = src.nodata
        print(f"  Rastergröße: {src.width:,} x {src.height:,}")

        for _, window in src.block_windows(1):
            data = src.read(1, window=window).astype(np.float32, copy=False)

            if nodata is not None:
                data = data[data != nodata]
            data = data[np.isfinite(data)]
            data = data[data > 0]

            if data.size:
                chunks.append(data)

    if not chunks:
        raise ValueError("Keine gültigen Rasterzellen gefunden.")

    values = np.concatenate(chunks)

    return {
        f"{prefix}_mean": float(np.mean(values, dtype=np.float64)),
        f"{prefix}_median": float(np.median(values)),
        f"{prefix}_std": float(np.std(values, dtype=np.float64)),
        f"{prefix}_p10": float(np.percentile(values, 10)),
        f"{prefix}_p90": float(np.percentile(values, 90)),
        f"{prefix}_min": float(values.min()),
        f"{prefix}_max": float(values.max()),
        f"{prefix}_valid_cells": int(values.size),
    }

# process_country function to download wind speed and power density data for a given country, 
# calculate statistics, and return results
def process_country(iso3, country):
    print()
    print("=" * 70)
    print(country)
    print("=" * 70)

    result = {"country": country, "iso_code": iso3, "height_m": HEIGHT}

    wind_url = f"{GWA_BASE_URL}/{iso3}/wind-speed/{HEIGHT}"
    wind_file = os.path.join(DOWNLOAD_FOLDER, f"{iso3}_wind_speed_{HEIGHT}m.tif")
    try:
        download_file(wind_url, wind_file)
        result.update(calculate_statistics(wind_file, "wind_speed_100m"))
        print("  Mittlere Windgeschwindigkeit:", round(result["wind_speed_100m_mean"], 3), "m/s")
    except Exception as e:
        print(f"  WIND-FEHLER: {e}")
        return None
    finally:
        if os.path.exists(wind_file):
            os.remove(wind_file)

    power_url = f"{GWA_BASE_URL}/{iso3}/power-density/{HEIGHT}"
    power_file = os.path.join(DOWNLOAD_FOLDER, f"{iso3}_power_density_{HEIGHT}m.tif")
    try:
        download_file(power_url, power_file)
        result.update(calculate_statistics(power_file, "power_density_100m"))
        print("  Mittlere Leistungsdichte:", round(result["power_density_100m_mean"], 2), "W/m²")
    except Exception as e:
        print(f"  LEISTUNGSDICHTE-FEHLER: {e}")
        # Land nicht verwerfen, Windgeschwindigkeit bleibt nutzbar
    finally:
        if os.path.exists(power_file):
            os.remove(power_file)

    return result


if os.path.exists(GWA_FILE):
    gwa_df = pd.read_csv(GWA_FILE)
    completed = set(gwa_df["iso_code"])
    print(f"Vorhandener Fortschritt gefunden: {len(completed)} Länder")
else:
    gwa_df = pd.DataFrame()
    completed = set()

for iso3, country in COUNTRIES.items():
    if iso3 in completed:
        print(f"ÜBERSPRINGE {country} (bereits erfasst)")
        continue

    result = process_country(iso3, country)
    if result is not None:
        gwa_df = pd.concat([gwa_df, pd.DataFrame([result])], ignore_index=True)
        gwa_df.to_csv(GWA_FILE, index=False, encoding="utf-8")
        print(f"  FORTSCHRITT GESPEICHERT: {len(gwa_df)}/{len(COUNTRIES)} Länder")
    else:
        print(f"  FEHLGESCHLAGEN: {country}")

    time.sleep(2)

if len(gwa_df) > 0:
    gwa_df = gwa_df.sort_values(
        "power_density_100m_mean", ascending=False, na_position="last"
    ).reset_index(drop=True)

    for column in gwa_df.select_dtypes(include="number").columns:
        gwa_df[column] = gwa_df[column].round(3)

    gwa_df.to_csv(GWA_FILE, index=False, encoding="utf-8")

print(f"\nFertig. {GWA_FILE} enthält {len(gwa_df)} Länder.")


In [ ]:
from io import StringIO

GENERATION_URL = (
    "https://ourworldindata.org/grapher/wind-generation.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)
CAPACITY_URL = (
    "https://ourworldindata.org/grapher/"
    "cumulative-installed-wind-energy-capacity-gigawatts.csv"
    "?v=1&csvType=full&useColumnShortNames=false"
)


def fetch_owid_csv(url: str) -> pd.DataFrame:
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    return pd.read_csv(StringIO(r.text))


gwa = pd.read_csv(GWA_FILE)
print(f"{len(gwa)} Länder im GWA-Datensatz:", gwa["country"].tolist())

print("\nLade Erzeugungsdaten...")
generation = fetch_owid_csv(GENERATION_URL)
print("Spalten (Erzeugung):", generation.columns.tolist())

print("\nLade Kapazitätsdaten...")
capacity = fetch_owid_csv(CAPACITY_URL)
print("Spalten (Kapazität):", capacity.columns.tolist())

if "Wind" not in generation.columns or "Wind" not in capacity.columns:
    raise ValueError(
        "OWID-Spaltenname 'Wind' nicht gefunden - OWID hat vermutlich das "
        f"Schema geändert. Erzeugung: {generation.columns.tolist()}, "
        f"Kapazität: {capacity.columns.tolist()}"
    )
generation = generation.rename(columns={"Wind": "wind_generation_twh"})
capacity = capacity.rename(columns={"Wind": "wind_capacity_gw"})

gen_latest = generation.sort_values("Year").groupby("Entity", as_index=False).last()
capacity_latest = capacity.sort_values("Year").groupby("Entity", as_index=False).last()

name_map = {
    "Turkey": "Türkiye",
    "United States of America": "United States",
}
for df_ in (gen_latest, capacity_latest):
    df_["Entity"] = df_["Entity"].replace(name_map)

merged = gwa.merge(gen_latest, left_on="country", right_on="Entity", how="left")
merged = merged.merge(
    capacity_latest,
    left_on="country",
    right_on="Entity",
    how="left",
    suffixes=("", "_irena"),
)
merged = merged.drop(columns=[c for c in ("Entity", "Entity_irena") if c in merged.columns])

missing = merged.loc[merged["Year"].isna(), "country"].tolist() if "Year" in merged.columns else []
if missing:
    print("\n⚠️  Keine Erzeugungs-/Kapazitätsdaten gefunden für:", missing)
    print("    (ggf. Ländernamen manuell in name_map ergänzen)")

merged.to_csv(MERGED_FILE, index=False)
print(f"\nGespeichert: {MERGED_FILE}  shape={merged.shape}")
print(merged.head())


In [ ]:
HOURS_PER_YEAR = 8760

df = pd.read_csv(MERGED_FILE)
print("Eingelesene Spalten:", df.columns.tolist())

RENAME_MAP = {
    "wind_speed_100m_mean": "wind_speed_100m_mean_ms",
    "wind_speed_100m_median": "wind_speed_100m_median_ms",
    "wind_speed_100m_std": "wind_speed_100m_std_ms",
    "wind_speed_100m_p10": "wind_speed_100m_p10_ms",
    "wind_speed_100m_p90": "wind_speed_100m_p90_ms",
    "wind_speed_100m_min": "wind_speed_100m_min_ms",
    "wind_speed_100m_max": "wind_speed_100m_max_ms",
    "wind_speed_100m_valid_cells": "wind_speed_valid_cells",
    "power_density_100m_mean": "power_density_100m_mean_wm2",
    "power_density_100m_median": "power_density_100m_median_wm2",
    "power_density_100m_std": "power_density_100m_std_wm2",
    "power_density_100m_p10": "power_density_100m_p10_wm2",
    "power_density_100m_p90": "power_density_100m_p90_wm2",
    "power_density_100m_min": "power_density_100m_min_wm2",
    "power_density_100m_max": "power_density_100m_max_wm2",
    "power_density_100m_valid_cells": "power_density_valid_cells",
}
df = df.rename(columns=RENAME_MAP)

for required_col in ("wind_generation_twh", "wind_capacity_gw"):
    if required_col not in df.columns:
        raise ValueError(
            f"Spalte '{required_col}' fehlt nach dem Merge. "
            f"Verfügbare Spalten: {df.columns.tolist()}"
        )

df["generation_year"] = df["Year"] if "Year" in df.columns else np.nan
if "Year_irena" in df.columns:
    df["capacity_year"] = df["Year_irena"]
elif "Year" in df.columns:
    df["capacity_year"] = df["Year"]
else:
    df["capacity_year"] = np.nan

df["generation_per_gw"] = df["wind_generation_twh"] / df["wind_capacity_gw"]
df["capacity_factor_pct"] = (
    df["wind_generation_twh"] * 1000 / (df["wind_capacity_gw"] * HOURS_PER_YEAR)
) * 100

FINAL_COLUMNS = [
    "country", "iso_code", "height_m",
    "wind_speed_100m_mean_ms", "wind_speed_100m_median_ms", "wind_speed_100m_std_ms",
    "wind_speed_100m_p10_ms", "wind_speed_100m_p90_ms",
    "wind_speed_100m_min_ms", "wind_speed_100m_max_ms", "wind_speed_valid_cells",
    "power_density_100m_mean_wm2", "power_density_100m_median_wm2", "power_density_100m_std_wm2",
    "power_density_100m_p10_wm2", "power_density_100m_p90_wm2",
    "power_density_100m_min_wm2", "power_density_100m_max_wm2", "power_density_valid_cells",
    "generation_year", "wind_generation_twh",
    "capacity_year", "wind_capacity_gw",
    "generation_per_gw", "capacity_factor_pct",
]

missing = [c for c in FINAL_COLUMNS if c not in df.columns]
if missing:
    raise ValueError(f"Diese Zielspalten fehlen nach der Verarbeitung: {missing}")

final_df = df[FINAL_COLUMNS].copy()
final_df.to_csv(FINAL_FILE, index=False)

print(f"\nGespeichert: {FINAL_FILE}")
print("Shape:", final_df.shape)
print(final_df.head())